In [1]:
%load_ext jupyter_black

In [2]:
import sys

sys.path.append("/home/ecm5702/dev/downscaling-tools")
sys.path.append("/home/ecm5702/dev/downscaling-tools/manual_inference")

In [3]:
import os

os.environ["DEV"] = "/home/ecm5702/dev/"
os.environ["DATA_DIR"] = "/home/mlx/ai-ml/datasets/"
os.environ["DATA_STABLE_DIR"] = "/home/mlx/ai-ml/datasets/stable/"
os.environ["OUTPUT"] = "/ec/res4/scratch/ecm5702/aifs"
os.environ["GRID_DIR"] = "/home/mlx/ai-ml/grids/"
os.environ["INTER_MAT_DIR"] = "/home/ecm5702/hpcperm/data/inter_mat"
os.environ["RESIDUAL_STATISTICS_DIR"] = (
    "/home/ecm5702/hpcperm/data/residuals_statistics/"
)
os.environ["GRAPHS"] = "/home/ecm5702/scratch/aifs/graphs"
os.environ["SLURM_GPUS_PER_NODE"] = "1"
os.environ["SLURM_NNODES"] = "1"

In [4]:
from get_objects_from_ckpt import ObjectFromCheckpointLoader

In [5]:
from get_objects_from_ckpt import *

In [50]:
### 100k trained model
dir_exp = "/home/ecm5702/scratch/aifs/checkpoint"
name_exp = "3aeb36edfa954fcf81f5d1db2d82b72f/"
name_ckpt = "anemoi-by_epoch-epoch_001-step_009248.ckpt"

In [69]:
# 3e5 like old config + FT 1e5 with o320_dict_0_72 at 100k steps
dir_exp = "/home/ecm5702/scratch/aifs/checkpoint"
name_exp = "0e1602b36722471f97a3dfa93b81297e"
name_ckpt = "anemoi-by_epoch-epoch_021-step_101728.ckpt"

In [108]:
# 3e5 like old config + FT 1e5 with o320_dict_0_72
dir_exp = "/home/ecm5702/scratch/aifs/checkpoint"
name_exp = "0e1602b36722471f97a3dfa93b81297e"
name_ckpt = "anemoi-by_epoch-epoch_021-step_100000.ckpt"

In [92]:
# o320 - run on leonardo with the old graph and the new dictionary, after DOUBLE finetuning
dir_exp = "/home/ecm5702/scratch/aifs/checkpoint"
name_exp = "ac7760efa687457da469f2705312a1c5/"
name_ckpt = "anemoi-by_epoch-epoch_021-step_101728.ckpt"

In [126]:
# o320 - run on leonardo with the old graph and the new dictionary, after one finetuning
dir_exp = "/home/ecm5702/scratch/aifs/checkpoint"
name_exp = "ec4d16fb6f8c402992e1e29ec7ddfc0e/"
name_ckpt = "anemoi-by_epoch-epoch_021-step_101728.ckpt"

In [129]:
# AC RUN
dir_exp = "/home/ecm5702/scratch/aifs/checkpoint"
name_exp = "5782221741d84602b7048f2d273c705a/"
name_ckpt = "last.ckpt"

In [137]:
# o320 - run on leonardo with the old graph and the new dictionary, before one finetuning
dir_exp = "/home/ecm5702/scratch/aifs/checkpoint"
name_exp = "dc2d9596540d420c95da56917577a445/"
name_ckpt = "anemoi-by_epoch-epoch_179-step_832320.ckpt"

In [6]:
# AG RUN
dir_exp = "/home/ecm5702/scratch/aifs/checkpoint"
name_exp = "8ef18bfef7804fcda809f032261a2d39/"
name_ckpt = "last.ckpt"

In [7]:
print(name_exp)
checkpoint, config_checkpoint = get_checkpoint(dir_exp, name_exp, name_ckpt)
config = instantiate_config()
config_checkpoint = adapt_config_hpc(config_checkpoint, config)

8ef18bfef7804fcda809f032261a2d39/


/home/ecm5702/dev/downscaling-tools/get_objects_from_ckpt.py:72: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  initialize_config_dir(config_dir=anemoi_config_dir, job_name="compose_config")


In [8]:
device = "cuda"
object_loader = ObjectFromCheckpointLoader(dir_exp, name_exp, name_ckpt)
object_loader.config_for_datamodule.dataloader.validation.frequency = "50h"
object_loader.load()
# object_loader.config_checkpoint: modify here if desired

datamodule = object_loader.datamodule
interface = object_loader.interface.to(device)
downscaler = object_loader.downscaler.to(device)
graph_data = object_loader.graph_data
inference_model = object_loader.inference_model.to(device)

/home/ecm5702/dev/downscaling-tools/get_objects_from_ckpt.py:72: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  initialize_config_dir(config_dir=anemoi_config_dir, job_name="compose_config")
/etc/ecmwf/nfs/dh2_home_a/ecm5702/dev/.ds-dyn/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/etc/ecmwf/nfs/dh2_home_a/ecm5702/dev/anemoi-datasets/src/anemoi/datasets/data/forwards.py:214: UserWarning: The behaviour of Zip.collect_supporting_arrays() is not well defined
  warnings.warn(f"The behaviour of {self.__class__.__name__}.collect_supporting_arrays() is not well defined")
/etc/ecmwf/nfs/dh2_home_a/ecm5702/dev/.ds-dyn/lib/python3.11/site-packages/pytorch_lightning/utilities/migration/utils.py:56: The load

### normal prediction

In [140]:
val_loader = datamodule.val_dataloader()
for batch_idx, batch in enumerate(val_loader):
    if batch_idx > 2:
        break
batch = [x.to(device) for x in batch]

In [141]:
extra_args = {"num_steps": 20, "S_max": 80, "sigma_max": 80}
output = inference_model.predict_step(batch[0], batch[1], extra_args=extra_args)
output.shape

noise_scheduler_params: {'schedule_type': 'karras', 'sigma_max': 80, 'sigma_min': 0.03, 'rho': 7.0, 'num_steps': 20}
sampler_params: {'sampler': 'heun', 'S_churn': 2.5, 'S_min': 0.75, 'S_max': 80, 'S_noise': 1.05}
x_in_interp val tensor(0.3322, device='cuda:0')
x_in_hres val tensor(0.1915, device='cuda:0')
y val tensor(9.6872, device='cuda:0')
sigmas tensor([8.0000e+01, 6.2081e+01, 4.7719e+01, 3.6304e+01, 2.7315e+01, 2.0305e+01,
        1.4897e+01, 1.0774e+01, 7.6708e+00, 5.3673e+00, 3.6842e+00, 2.4754e+00,
        1.6238e+00, 1.0368e+00, 6.4192e-01, 3.8368e-01, 2.2015e-01, 1.2040e-01,
        6.2206e-02, 3.0000e-02, 0.0000e+00], device='cuda:0',
       dtype=torch.float64) 2.5 0.75 80 1.05


/etc/ecmwf/nfs/dh2_home_a/ecm5702/dev/anemoi-core/models/src/anemoi/models/models/downscaler_encoder_processor_decoder.py:583: UserWarning: noise_scheduler_config: {'schedule_type': 'karras', 'sigma_max': 80, 'sigma_min': 0.03, 'rho': 7.0, 'num_steps': 20}
  warnings.warn(f"noise_scheduler_config: {noise_scheduler_config}")
/etc/ecmwf/nfs/dh2_home_a/ecm5702/dev/anemoi-core/models/src/anemoi/models/models/downscaler_encoder_processor_decoder.py:628: UserWarning: diffusion_sampler_config: {'sampler': 'heun', 'S_churn': 2.5, 'S_min': 0.75, 'S_max': 80, 'S_noise': 1.05}
  warnings.warn(f"diffusion_sampler_config: {diffusion_sampler_config}")


torch.Size([1, 1, 1, 421120, 68])

In [142]:
name_to_index_output = datamodule.data_indices.name_to_index_output
for weather_state in ["2t", "10u", "10v", "z_500"]:
    t = output[..., name_to_index_output[weather_state]]
    mean = t.mean().item()
    min_ = t.min().item()
    max_ = t.max().item()
    print(f"{weather_state}: mean={mean:.3f}, min={min_:.3f}, max={max_:.3f}")

2t: mean=287.288, min=225.778, max=310.002
10u: mean=-0.293, min=-18.802, max=22.794
10v: mean=0.253, min=-24.054, max=23.448
z_500: mean=55616.203, min=46602.918, max=57977.074


In [143]:
name_to_index_output = datamodule.data_indices.name_to_index_output
for weather_state in ["2t", "10u", "10v", "z_500"]:
    t = batch[2][..., name_to_index_output[weather_state]]
    mean = t.mean().item()
    min_ = t.min().item()
    max_ = t.max().item()
    print(f"{weather_state}: mean={mean:.3f}, min={min_:.3f}, max={max_:.3f}")

2t: mean=287.329, min=224.610, max=309.713
10u: mean=-0.448, min=-19.791, max=24.564
10v: mean=0.026, min=-22.718, max=23.343
z_500: mean=55643.289, min=46623.426, max=58001.176


#### save spectra

In [145]:
### to use with save_spectra_from_grib that will take the grib in input and compare it to references

In [144]:
import earthkit.data as ekd
import numpy as np
from earthkit.data import FieldList
import os
import xarray as xr

field_10u = output[0, 0, 0, :, name_to_index_output["10u"]].cpu()
field_10v = output[0, 0, 0, :, name_to_index_output["10v"]].cpu()
field_2t = output[0, 0, 0, :, name_to_index_output["2t"]].cpu()

working_dir = "/home/ecm5702/dev/notebooks/debug-anemoi-core-merge"

template_path = os.path.join(working_dir, "o320-template.grib")
out_path = os.path.join(working_dir, "o320_for_spectra.grib")

# --- ensure clean output file ---
if os.path.exists(out_path):
    os.remove(out_path)

ds = ekd.from_source("file", template_path)
md_2t = ds[0].metadata()

arr_2t = np.array(field_2t, dtype=float)
arr_10u = np.array(field_10u, dtype=float)
arr_10v = np.array(field_10v, dtype=float)

fl_2t = FieldList.from_array(arr_2t, md_2t)
fl_10u = FieldList.from_array(arr_10u, md_2t.override(shortName="10u"))
fl_10v = FieldList.from_array(arr_10v, md_2t.override(shortName="10v"))

fl_all = fl_2t + fl_10u + fl_10v

try:
    fl_all.to_target("file", out_path)
except Exception:
    fl_all.save(out_path)

print("Written:", out_path)
ekd.from_source("file", out_path).ls()

Written: /home/ecm5702/dev/notebooks/debug-anemoi-core-merge/o320_for_spectra.grib


,centre,shortName,typeOfLevel,level,dataDate,dataTime,stepRange,dataType,number,gridType
0,ecmf,2t,surface,0,20250201,0,0,fc,0,reduced_gg
1,ecmf,10u,surface,0,20250201,0,0,fc,0,reduced_gg
2,ecmf,10v,surface,0,20250201,0,0,fc,0,reduced_gg


### predict step with intermediates

In [59]:
import types
import torch


def patch_model_sample_to_return_intermediates(model, *, cpu=True):
    """
    After patch:
      model.sample(...) -> returns (y_final, intermediates)

    intermediates: list of dicts
      intermediates[i] = {
          "step": i,
          "y": full state tensor after diffusion step i
      }
    """
    orig_sample = model.sample

    @torch.no_grad()
    def patched_sample(
        self,
        x_in_interp_to_hres: torch.Tensor,
        x_in_hres: torch.Tensor,
        model_comm_group=None,
        grid_shard_shapes=None,
        noise_scheduler_params=None,
        sampler_params=None,
        **kwargs,
    ):
        import anemoi.models.samplers.diffusion_samplers as diffusion_samplers

        # -------------------------
        # Noise schedule
        # -------------------------
        noise_scheduler_config = dict(self.inference_defaults.noise_scheduler)
        if noise_scheduler_params is not None:
            noise_scheduler_config.update(noise_scheduler_params)

        schedule_type = noise_scheduler_config.pop("schedule_type")
        scheduler_cls = diffusion_samplers.NOISE_SCHEDULERS[schedule_type]
        scheduler = scheduler_cls(**noise_scheduler_config)
        sigmas = scheduler.get_schedule(x_in_interp_to_hres.device, torch.float64)

        # -------------------------
        # Initial noise
        # -------------------------
        batch_size, ensemble_size, grid_size = (
            x_in_interp_to_hres.shape[0],
            x_in_interp_to_hres.shape[2],
            x_in_interp_to_hres.shape[-2],
        )
        time_size = 1
        shape = (
            batch_size,
            time_size,
            ensemble_size,
            grid_size,
            self.num_output_channels,
        )
        y = torch.randn(shape, device=x_in_interp_to_hres.device) * sigmas[0]

        # -------------------------
        # Sampler config
        # -------------------------
        diffusion_sampler_config = dict(self.inference_defaults.diffusion_sampler)
        if sampler_params is not None:
            diffusion_sampler_config.update(sampler_params)

        sampler_name = diffusion_sampler_config.pop("sampler")
        sampler_cls = diffusion_samplers.DIFFUSION_SAMPLERS[sampler_name]
        sampler = sampler_cls(dtype=sigmas.dtype, **diffusion_sampler_config)

        # -------------------------
        # Non-Heun fallback
        # -------------------------
        if sampler_name != "heun":
            y_final = sampler.sample(
                x_in_interp_to_hres,
                x_in_hres,
                y,
                sigmas,
                self.fwd_with_preconditioning,
                grid_shard_shapes=grid_shard_shapes,
                model_comm_group=model_comm_group,
            )
            return y_final, []

        # -------------------------
        # Heun loop with intermediates
        # -------------------------
        intermediates = []

        def store(step, y_tensor):
            yy = y_tensor.detach().clone()
            if cpu:
                yy = yy.cpu()
            intermediates.append(
                {
                    "step": step,
                    "y": yy,
                }
            )

        S_churn = diffusion_sampler_config.get("S_churn", sampler.S_churn)
        S_min = diffusion_sampler_config.get("S_min", sampler.S_min)
        S_max = diffusion_sampler_config.get("S_max", sampler.S_max)
        S_noise = diffusion_sampler_config.get("S_noise", sampler.S_noise)
        dtype = getattr(sampler, "dtype", sigmas.dtype)
        eps_prec = getattr(sampler, "eps_prec", 1e-10)

        batch_size, ensemble_size = (
            x_in_interp_to_hres.shape[0],
            x_in_interp_to_hres.shape[2],
        )
        num_steps = len(sigmas) - 1

        # store initial state (step 0)
        store(0, y)

        for i in range(num_steps):
            sigma_i = sigmas[i]
            sigma_next = sigmas[i + 1]

            apply_churn = (S_min <= sigma_i <= S_max) and (S_churn > 0.0)
            if apply_churn:
                gamma = min(
                    S_churn / num_steps,
                    torch.sqrt(
                        torch.tensor(2.0, dtype=sigma_i.dtype, device=sigma_i.device)
                    )
                    - 1,
                )
                sigma_effective = sigma_i + gamma * sigma_i
                epsilon = torch.randn_like(y) * S_noise
                y = y + torch.sqrt(sigma_effective**2 - sigma_i**2) * epsilon
            else:
                sigma_effective = sigma_i

            D1 = self.fwd_with_preconditioning(
                x_in_interp_to_hres,
                x_in_hres,
                y.to(dtype=x_in_interp_to_hres.dtype),
                sigma_effective.view(1, 1, 1, 1)
                .expand(batch_size, ensemble_size, 1, 1)
                .to(x_in_interp_to_hres.dtype),
                model_comm_group,
                grid_shard_shapes,
            ).to(dtype)

            d = (y - D1) / (sigma_effective + eps_prec)
            y_next = y + (sigma_next - sigma_effective) * d

            if sigma_next > eps_prec:
                D2 = self.fwd_with_preconditioning(
                    x_in_interp_to_hres,
                    x_in_hres,
                    y_next.to(dtype=x_in_interp_to_hres.dtype),
                    sigma_next.view(1, 1, 1, 1)
                    .expand(batch_size, ensemble_size, 1, 1)
                    .to(dtype=x_in_interp_to_hres.dtype),
                    model_comm_group,
                    grid_shard_shapes,
                ).to(dtype)
                d_prime = (y_next - D2) / (sigma_next + eps_prec)
                y = y + (sigma_next - sigma_effective) * (d + d_prime) / 2
            else:
                y = y_next

            # store state AFTER this diffusion step
            store(i + 1, y)

        return y, intermediates

    model.sample = types.MethodType(patched_sample, model)
    model._orig_sample = orig_sample
    return model

In [61]:
patch_model_sample_to_return_intermediates(inference_model.model, cpu=True)

AnemoiDownscalingModelEncProcDec(
  (node_attributes): NamedNodesAttributes(
    (trainable_tensors): ModuleDict(
      (data): TrainableTensor()
      (hidden): TrainableTensor()
      (lres): TrainableTensor()
    )
  )
  (encoder): GraphTransformerForwardMapper(
    (activation): GELU(approximate='none')
    (trainable): TrainableTensor()
    (proc): GraphTransformerMapperBlock(
      (lin_key): Linear(in_features=512, out_features=512, bias=True)
      (lin_query): Linear(in_features=512, out_features=512, bias=True)
      (lin_value): Linear(in_features=512, out_features=512, bias=True)
      (lin_self): Linear(in_features=512, out_features=512, bias=True)
      (lin_edge): Linear(in_features=3, out_features=512, bias=True)
      (q_norm): AutocastLayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (k_norm): AutocastLayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (conv): GraphTransformerConv()
      (projection): Linear(in_features=512, out_features=512, bias=T

In [86]:
@torch.no_grad()
def predict_step_with_intermediates(
    model,
    x_in_lres: torch.Tensor,
    x_in_hres: torch.Tensor,
    pre_processors,
    post_processors,
    multi_step: int,
    model_comm_group=None,
    gather_out=True,
    noise_scheduler_params=None,
    sampler_params=None,
    pre_processors_tendencies=None,
    post_processors_tendencies=None,
    **kwargs,
):

    noise_scheduler_params = {
        "schedule_type": kwargs.get("schedule_type", "karras"),
        "sigma_max": kwargs.get("sigma_max", 100000),
        "sigma_min": kwargs.get("sigma_min", 0.03),
        "rho": kwargs.get("rho", 7.0),
        "num_steps": kwargs.get("num_steps", 80),
    }

    sampler_params = {
        "sampler": kwargs.get("sampler", "heun"),
        "S_churn": kwargs.get("S_churn", 2.5),
        "S_min": kwargs.get("S_min", 0.75),
        "S_max": kwargs.get("S_max", 100000),
        "S_noise": kwargs.get("S_noise", 1.05),
    }

    print(noise_scheduler_params, sampler_params)

    if len(x_in_lres.shape) == 4:
        x_in_lres = x_in_lres[:, None, ...]
    if len(x_in_hres.shape) == 4:
        x_in_hres = x_in_hres[:, None, ...]

    before_sampling_data, grid_shard_shapes = model._before_sampling(
        x_in_lres,
        x_in_hres,
        pre_processors,
        multi_step,
        model_comm_group,
        pre_processors_tendencies=pre_processors_tendencies,
        post_processors_tendencies=post_processors_tendencies,
        **kwargs,
    )

    x_in_interp_to_hres = before_sampling_data[0]
    x_in_hres2 = before_sampling_data[1]

    y, intermediate_states = model.sample(
        x_in_interp_to_hres,
        x_in_hres2,
        model_comm_group=model_comm_group,
        grid_shard_shapes=grid_shard_shapes,
        noise_scheduler_params=noise_scheduler_params,
        sampler_params=sampler_params,
        **kwargs,
    )
    y = y.to(x_in_interp_to_hres.dtype)

    processed_intermediates = []

    for item in intermediate_states:
        step = item["step"]
        y_raw = item["y"]

        y_phys = model._after_sampling(
            y_raw.to(output.device),
            post_processors,
            before_sampling_data,
            model_comm_group,
            grid_shard_shapes,
            gather_out,
        )

        processed_intermediates.append(
            {
                "step": step,
                "y": y_phys.cpu(),
            }
        )

    out = model._after_sampling(
        y,
        post_processors,
        before_sampling_data,
        model_comm_group,
        grid_shard_shapes,
        gather_out,
        pre_processors_tendencies=pre_processors_tendencies,
        post_processors_tendencies=post_processors_tendencies,
        **kwargs,
    )

    return out, processed_intermediates

In [87]:
val_loader = datamodule.val_dataloader()
for batch_idx, batch in enumerate(val_loader):
    if batch_idx > 2:
        break
batch = [x.to(device) for x in batch]

In [88]:
x_in_lres = batch[0]
x_in_hres = batch[1]

In [94]:
output, intermediate_states = predict_step_with_intermediates(
    inference_model.model,
    x_in_lres=x_in_lres,
    x_in_hres=x_in_hres,
    pre_processors=inference_model.pre_processors,
    post_processors=inference_model.post_processors,
    multi_step=inference_model.multi_step,
    extra_args=extra_args,
    num_steps=20,
    S_max=80,
    sigma_max=80,
)

{'schedule_type': 'karras', 'sigma_max': 80, 'sigma_min': 0.03, 'rho': 7.0, 'num_steps': 20} {'sampler': 'heun', 'S_churn': 2.5, 'S_min': 0.75, 'S_max': 80, 'S_noise': 1.05}


In [95]:
len(intermediate_states)

21

In [97]:
idx_10v = datamodule.data_indices.name_to_index_output["10v"]

for item in intermediate_states:
    step = item["step"]
    y = item["y"]  # full tensor at this diffusion step

    t = y[..., idx_10v]
    mean = t.mean().item()
    min_ = t.min().item()
    max_ = t.max().item()

    print(f"step {step:03d} | " f"10v: mean={mean:.3f}, min={min_:.3f}, max={max_:.3f}")

step 000 | 10v: mean=0.112, min=-374.148, max=315.330
step 001 | 10v: mean=0.096, min=-286.217, max=289.559
step 002 | 10v: mean=0.064, min=-227.006, max=214.047
step 003 | 10v: mean=0.036, min=-175.367, max=165.909
step 004 | 10v: mean=0.005, min=-133.323, max=135.080
step 005 | 10v: mean=0.008, min=-97.293, max=104.015
step 006 | 10v: mean=0.013, min=-74.041, max=79.825
step 007 | 10v: mean=0.004, min=-52.226, max=56.258
step 008 | 10v: mean=0.018, min=-40.700, max=44.097
step 009 | 10v: mean=0.020, min=-33.079, max=33.875
step 010 | 10v: mean=0.029, min=-31.460, max=29.142
step 011 | 10v: mean=0.037, min=-27.449, max=26.343
step 012 | 10v: mean=0.037, min=-25.927, max=24.886
step 013 | 10v: mean=0.035, min=-25.267, max=23.940
step 014 | 10v: mean=0.033, min=-25.684, max=23.058
step 015 | 10v: mean=0.032, min=-25.540, max=22.888
step 016 | 10v: mean=0.032, min=-25.497, max=22.813
step 017 | 10v: mean=0.033, min=-25.488, max=22.785
step 018 | 10v: mean=0.034, min=-25.489, max=22.776
s

In [98]:
name_to_index_output = datamodule.data_indices.name_to_index_output
for weather_state in ["2t", "10u", "10v", "z_500"]:
    t = output[..., name_to_index_output[weather_state]]
    mean = t.mean().item()
    min_ = t.min().item()
    max_ = t.max().item()
    print(f"{weather_state}: mean={mean:.3f}, min={min_:.3f}, max={max_:.3f}")

2t: mean=287.252, min=225.692, max=309.911
10u: mean=-0.521, min=-19.045, max=22.760
10v: mean=0.034, min=-25.490, max=22.771
z_500: mean=55633.414, min=46513.121, max=58005.121


In [54]:
name_to_index_output = datamodule.data_indices.name_to_index_output
for weather_state in ["2t", "10u", "10v", "z_500"]:
    t = batch[2][..., name_to_index_output[weather_state]]
    mean = t.mean().item()
    min_ = t.min().item()
    max_ = t.max().item()
    print(f"{weather_state}: mean={mean:.3f}, min={min_:.3f}, max={max_:.3f}")

2t: mean=287.329, min=224.610, max=309.713
10u: mean=-0.448, min=-19.791, max=24.564
10v: mean=0.026, min=-22.718, max=23.343
z_500: mean=55643.289, min=46623.426, max=58001.176


In [106]:
import numpy as np
import xarray as xr
import torch


def intermediates_to_xarray(
    *,
    x_in_lres,  # torch.Tensor or np.ndarray
    y_true,  # torch.Tensor or np.ndarray (ground truth, optional but you said you have y)
    y_pred,  # torch.Tensor or np.ndarray (final output, after _after_sampling)
    intermediate_states,  # list of {"step": int, "y": tensor} (ideally already post-processed)
    data_indices,  # datamodule.data_indices (needs name_to_index_output)
    attrs=None,
):
    """
    Build an in-memory xr.Dataset that follows your nomenclature:
      - x
      - y
      - y_pred
      - intermediates (step dimension)

    Dimensions:
      sample, member, step, time, grid, var

    Notes:
      - This assumes tensors are shaped like (sample, member, time, grid, var) OR broadcastable.
      - If your tensors are missing sample/member/time dims, they will be added as size-1 dims.
    """

    def _to_np(a):
        if a is None:
            return None
        if torch.is_tensor(a):
            return a.detach().cpu().numpy()
        return np.asarray(a)

    def _ensure_5d(arr):
        # target: (sample, member, time, grid, var)
        if arr is None:
            return None
        arr = _to_np(arr)
        if arr.ndim == 5:
            return arr
        if (
            arr.ndim == 4
        ):  # (member,time,grid,var) or (time,member,grid,var) ambiguity; assume (time,member,grid,var) is unlikely
            return arr[None, ...]
        if arr.ndim == 3:  # (grid,var) with time missing? assume (time=1,grid,var)
            return arr[None, None, None, ...]
        if arr.ndim == 2:  # (grid,var)
            return arr[None, None, None, ...]
        raise ValueError(
            f"Unsupported ndim={arr.ndim} for array with shape {arr.shape}"
        )

    x5 = _ensure_5d(x_in_lres)
    y5 = _ensure_5d(y_true) if y_true is not None else None
    yp5 = _ensure_5d(y_pred)

    # intermediates: (step, sample, member, time, grid, var)
    steps = [int(it["step"]) for it in intermediate_states]
    ys = [_ensure_5d(it["y"]) for it in intermediate_states]
    inter6 = np.stack(ys, axis=0)

    # var names
    name_to_index_output = data_indices.name_to_index_output
    nvar = max(name_to_index_output.values()) + 1
    var_names = [None] * nvar
    for n, idx in name_to_index_output.items():
        var_names[idx] = n
    var_names = [v if v is not None else f"var_{i}" for i, v in enumerate(var_names)]

    coords = {
        "step": np.array(steps, dtype=int),
        "sample": np.arange(x5.shape[0]),
        "member": np.arange(x5.shape[1]),
        "time": np.arange(x5.shape[2]),
        "grid": np.arange(x5.shape[3]),
        "var": np.array(var_names),
    }

    data_vars = {
        "x": (("sample", "member", "time", "grid", "var"), x5),
        "y_pred": (("sample", "member", "time", "grid", "var"), yp5),
        "intermediates": (("step", "sample", "member", "time", "grid", "var"), inter6),
    }
    if y5 is not None:
        data_vars["y"] = (("sample", "member", "time", "grid", "var"), y5)

    ds = xr.Dataset(data_vars=data_vars, coords=coords, attrs=attrs or {})
    return ds

In [107]:
ds = intermediates_to_xarray(
    x_in_lres=x_in_lres,                 # your input
    y_true=y,                            # your target (optional)
    y_pred=output,                       # final prediction
    intermediate_states=processed_intermediates,  # list of {"step","y"} in physical space
    data_indices=datamodule.data_indices,
    attrs={"grid": "O320"},
)

# Now use your existing plotting utilities on ds
# e.g. pass ds into get_region_ds / plot_x_y, or adapt them to read ds["intermediates"] by step


ValueError: need at least one array to stack

### other